In [ ]:
FUNCTION Alpha_Trimmed_Mean_Filter(image, kernel_size, alpha):
    # Step 1: Get the dimensions of the image
    height ← number of rows in image
    width ← number of columns in image

    # Step 2: Create an empty output image of same size
    output_image ← empty matrix of size (height, width)

    # Step 3: Compute padding size
    pad ← floor(kernel_size / 2)

    # Step 4: Pad the image using border replication
    padded_image ← Pad image by 'pad' pixels on all sides

    # Step 5: Traverse through each pixel in the image
    FOR i from 0 to height - 1 DO:
        FOR j from 0 to width - 1 DO:

            # Step 6: Extract the kernel window centered at (i, j)
            window ← Extract (kernel_size × kernel_size) region from padded_image

            # Step 7: Convert window into a 1D list and sort values
            sorted_values ← Sort window values in ascending order

            # Step 8: Trim alpha/2 smallest and alpha/2 largest values
            trimmed_values ← Remove first (alpha/2) values and last (alpha/2) values from sorted_values

            # Step 9: Compute mean of remaining values
            filtered_value ← Mean of trimmed_values

            # Step 10: Assign the computed value to output image
            output_image[i, j] ← filtered_value

    # Step 11: Return the filtered image
    RETURN output_image


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Load Image in Grayscale
image = cv2.imread('sample_image.jpg', 0)
rows, cols = image.shape

# Perform Fourier Transform
dft = np.fft.fft2(image)
dft_shift = np.fft.fftshift(dft)  # Shift zero frequency to the center

# Create Ideal Low-Pass and High-Pass Filters
D0 = 50  # Cutoff frequency (change for different effects)

# Create a filter mask
low_pass_mask = np.zeros((rows, cols), np.uint8)
high_pass_mask = np.ones((rows, cols), np.uint8)

# Center Coordinates
cx, cy = rows//2, cols//2

# Apply Low-Pass Filter: Set values inside cutoff radius to 1
for i in range(rows):
    for j in range(cols):
        if np.sqrt((i - cx)**2 + (j - cy)**2) < D0:
            low_pass_mask[i, j] = 1

# Apply High-Pass Filter: Set values inside cutoff radius to 0
high_pass_mask = 1 - low_pass_mask

# Apply Filters
low_pass_result = dft_shift * low_pass_mask
high_pass_result = dft_shift * high_pass_mask

# Inverse Fourier Transform
def inverse_fourier(transform):
    return np.abs(np.fft.ifft2(np.fft.ifftshift(transform)))

# Get Filtered Images
image_lpf = inverse_fourier(low_pass_result)
image_hpf = inverse_fourier(high_pass_result)

# Plot Results
plt.figure(figsize=(12,6))

plt.subplot(1,3,1), plt.imshow(image, cmap='gray'), plt.title("Original Image")
plt.subplot(1,3,2), plt.imshow(image_lpf, cmap='gray'), plt.title("Low-Pass Filtered Image")
plt.subplot(1,3,3), plt.imshow(image_hpf, cmap='gray'), plt.title("High-Pass Filtered Image")

plt.show()


In [ ]:
import cv2
import numpy as np

def adaptive_mean_filter(image, window_size=3):
    # Convert image to float
    img = np.float32(image)
    
    # Compute global variance
    global_variance = np.var(img)

    # Define padding for border handling
    pad_size = window_size // 2
    padded_img = cv2.copyMakeBorder(img, pad_size, pad_size, pad_size, pad_size, cv2.BORDER_REFLECT)

    # Output image
    filtered_img = np.zeros_like(img)

    # Apply adaptive mean filter
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            # Extract local window
            local_window = padded_img[i:i+window_size, j:j+window_size]

            # Compute local mean and variance
            local_mean = np.mean(local_window)
            local_variance = np.var(local_window)

            # Compute adaptive mean filter
            if local_variance == 0:
                filtered_img[i, j] = local_mean  # Prevent division by zero
            else:
                filtered_img[i, j] = image[i, j] - (global_variance / local_variance) * (image[i, j] - local_mean)

    return np.uint8(filtered_img)

# Load a noisy image
image = cv2.imread('noisy_image.jpg', cv2.IMREAD_GRAYSCALE)

# Apply the adaptive mean filter
filtered_image = adaptive_mean_filter(image)

# Show results
cv2.imshow("Original", image)
cv2.imshow("Adaptive Mean Filter", filtered_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
def adaptive_local_variance_filter(image, window_size=3):
    img = np.float32(image)
    
    # Compute global variance
    global_variance = np.var(img)

    # Define padding
    pad_size = window_size // 2
    padded_img = cv2.copyMakeBorder(img, pad_size, pad_size, pad_size, pad_size, cv2.BORDER_REFLECT)

    # Output image
    filtered_img = np.zeros_like(img)

    # Apply adaptive local variance filter
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            # Extract local window
            local_window = padded_img[i:i+window_size, j:j+window_size]

            # Compute local mean and variance
            local_mean = np.mean(local_window)
            local_variance = np.var(local_window)

            # Compute adaptive variance filter
            if local_variance == 0:
                filtered_img[i, j] = local_mean  # Avoid division by zero
            else:
                filtered_img[i, j] = local_mean + (global_variance / local_variance) * (image[i, j] - local_mean)

    return np.uint8(filtered_img)

# Load a noisy image
image = cv2.imread('noisy_image.jpg', cv2.IMREAD_GRAYSCALE)

# Apply the adaptive local variance filter
filtered_image = adaptive_local_variance_filter(image)

# Show results
cv2.imshow("Original", image)
cv2.imshow("Adaptive Local Variance Filter", filtered_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

def geometric_mean_filter(image, kernel_size=3):
    img_pad = np.pad(image, ((kernel_size//2, kernel_size//2), (kernel_size//2, kernel_size//2)), mode='constant', constant_values=1)
    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = img_pad[i:i+kernel_size, j:j+kernel_size]
            product = np.prod(window, dtype=np.float64)  # Multiplying all pixels
            output[i, j] = product ** (1.0 / (kernel_size * kernel_size))  # Taking N-th root

    return np.uint8(output)

# Load an image in grayscale
image = cv2.imread('image.jpg', cv2.IMREAD_GRAYSCALE)
filtered_image = geometric_mean_filter(image, kernel_size=3)

cv2.imshow('Geometric Mean Filter', filtered_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
def harmonic_mean_filter(image, kernel_size=3):
    img_pad = np.pad(image, ((kernel_size//2, kernel_size//2), (kernel_size//2, kernel_size//2)), mode='constant', constant_values=1)
    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = img_pad[i:i+kernel_size, j:j+kernel_size]
            sum_reciprocal = np.sum(1.0 / (window + 1e-6))  # Avoid division by zero
            output[i, j] = (kernel_size * kernel_size) / sum_reciprocal

    return np.uint8(output)

filtered_image = harmonic_mean_filter(image, kernel_size=3)
cv2.imshow('Harmonic Mean Filter', filtered_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
def contra_harmonic_mean_filter(image, kernel_size=3, Q=1.5):
    img_pad = np.pad(image, ((kernel_size//2, kernel_size//2), (kernel_size//2, kernel_size//2)), mode='constant', constant_values=1)
    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            window = img_pad[i:i+kernel_size, j:j+kernel_size]
            num = np.sum(window ** (Q + 1))
            denom = np.sum(window ** Q) + 1e-6  # Avoid division by zero
            output[i, j] = num / denom

    return np.uint8(output)

filtered_image = contra_harmonic_mean_filter(image, kernel_size=3, Q=1.5)
cv2.imshow('Contra-harmonic Mean Filter', filtered_image)
cv2.waitKey(0)
cv2.destroyAllWindows()
